## 0. Environment Setup and Data Download

This cell checks if the notebook is running in Google Colab and, if so, downloads the necessary raw data files from a GitHub repository to ensure reproducibility.

In [ ]:
import os

# If running in Google Colab, download the specific raw files via wget
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    print("Setting up Colab environment and downloading raw data...")

    # Create the target directory
    !mkdir -p ./data/raw

    # The base URL for raw files in your GitHub repository
    base_url = "https://raw.githubusercontent.com/mlresearcher81/HCT-Microbiome-Benchmark/main/data/raw/"

    # List of all required raw files
    raw_files = [
        "tbltemperature.csv",
        "tblwbc.csv",
        "tblASVsamples.csv",
        "tblhctmeta.csv",
        "tblcounts_asv_melt.csv",
        "tblASVtaxonomy_silva132_v4v5_filter.csv"
    ]

    # Download each file if it doesn't already exist
    for file in raw_files:
        file_path = f"./data/raw/{file}"
        if not os.path.exists(file_path):
            file_url = base_url + file
            !wget -q -O {file_path} {file_url}

    print("Downloads complete! Ready to process.")

## 1. Import Libraries

In [ ]:
import pandas as pd  # Import the pandas library for data manipulation and analysis, commonly used for tabular data.
import numpy as np   # Import the numpy library for numerical operations, especially useful for array manipulation.

## 2. Load Raw Datasets

In [ ]:
# Load various datasets from specified CSV files into pandas DataFrames.
# The 'low_memory=False' option is used to prevent DtypeWarning for mixed types in columns,
# ensuring proper type inference for large files.
temperature_df = pd.read_csv("./data/raw/tbltemperature.csv", low_memory=False)  # Contains patient temperature records.
neutrophils_df = pd.read_csv("./data/raw/tblwbc.csv", low_memory=False)           # Contains white blood cell (WBC) counts, including neutrophils.
stool_df = pd.read_csv("./data/raw/tblASVsamples.csv")         # Contains stool consistency data and microbiome sample metadata.
allo_hct = pd.read_csv("./data/raw/tblhctmeta.csv")            # Contains metadata for allogeneic hematopoietic cell transplantation patients.

## 3. Initial Data Inspection

In [ ]:
# Display a concise summary of the 'temperature_df' DataFrame.
# This includes the index dtype, column dtypes, non-null values, and memory usage.
# It's a quick way to check for missing values and data types before further processing.
temperature_df.info()

### Inspect Neutrophils Data

### Inspect Stool Data

In [ ]:
# Display a concise summary of the 'neutrophils_df' DataFrame.
# This helps in understanding the structure, data types, and completeness of the neutrophil data.
neutrophils_df.info()

In [ ]:
# Display a concise summary of the 'stool_df' DataFrame.
# This provides an overview of the stool consistency data, including column types and non-null counts.
stool_df.info()

### Inspect Allo-HCT Data

In [ ]:
# Display a concise summary of the 'allo_hct' DataFrame.
# This gives insights into the patient metadata related to allogeneic HCT, such as patient IDs and disease types.
allo_hct.info()

## 4. Preprocessing for Merging

In [ ]:
# Convert the 'PatientID' column to string type across all relevant DataFrames.
# This standardizes the PatientID format, which is crucial for accurate and consistent merging operations,
# preventing potential errors due to mixed data types in identifier columns.
temperature_df['PatientID'] = temperature_df['PatientID'].astype(str)
neutrophils_df['PatientID'] = neutrophils_df['PatientID'].astype(str)
stool_df['PatientID'] = stool_df['PatientID'].astype(str)
allo_hct['PatientID'] = allo_hct['PatientID'].astype(str)

### Filter and Select Relevant Columns

In [ ]:
# Select only the essential columns from 'temperature_df' to reduce memory usage and focus on relevant data.
temperature_df = temperature_df[['PatientID', 'DayRelativeToNearestHCT', 'MaxTemperature']]

# Filter 'neutrophils_df' to include only rows where 'BloodCellType' is 'Neutrophils',
# then select 'PatientID', 'DayRelativeToNearestHCT', and 'Value'.
# This isolates the neutrophil-specific data for further analysis.
neutrophils_df = neutrophils_df[neutrophils_df['BloodCellType'] == 'Neutrophils'][['PatientID', 'DayRelativeToNearestHCT', 'Value']]

# Select essential columns from 'stool_df' for consistency information.
stool_df = stool_df[['PatientID', 'DayRelativeToNearestHCT', 'Consistency']]

# Select essential columns from 'allo_hct' for patient metadata.
allo_hct = allo_hct[['PatientID', 'TimepointOfTransplant', 'Disease']]

### Rename Columns for Consistency

In [ ]:
# Rename the 'Value' column in 'neutrophils_df' to 'NeutrophilCount'.
# This improves clarity and consistency in column naming conventions across the merged dataset,
# making the data more understandable.
neutrophils_df.rename(columns={'Value': 'NeutrophilCount'}, inplace=True)

## 5. Merge Clinical Datasets

In [ ]:
# Perform an initial merge of 'allo_hct' with 'temperature_df' based on 'PatientID'.
# An 'outer' join is used to ensure that all records from both DataFrames are kept, filling with NaN where no match exists.
merged_df = pd.merge(allo_hct, temperature_df, on=['PatientID'], how='outer')

# Merge the result with 'neutrophils_df' using both 'PatientID' and 'DayRelativeToNearestHCT'.
# This step combines neutrophil data with the existing merged clinical information.
merged_df = pd.merge(merged_df, neutrophils_df, on=['PatientID', 'DayRelativeToNearestHCT'], how='outer')

# Finally, merge with 'stool_df' using 'PatientID' and 'DayRelativeToNearestHCT'.
# This integrates stool consistency data into the comprehensive clinical dataset.
merged_df = pd.merge(merged_df, stool_df, on=['PatientID', 'DayRelativeToNearestHCT'], how='outer')

### Display Merged Data Sample

In [ ]:
# Display the first 5 rows of the 'merged_df' DataFrame.
# This allows for a quick visual inspection of the merged data structure, column alignment, and initial values.
merged_df.head(5)

## 6. Align Data to a Uniform Timeline

In [ ]:
# Define the minimum day for the uniform timeline, relative to the nearest HCT event.
# For example, -15 indicates 15 days before the transplant.
min_day = -15

# Define the maximum day for the uniform timeline, relative to the nearest HCT event.
# For example, 35 indicates 35 days after the transplant.
max_day = 35

### Prepare Timeline and Identify Unique Patients

In [ ]:
# Create a DataFrame 'all_days' containing a sequence of integers from 'min_day' to 'max_day' (inclusive).
# This DataFrame represents the standardized timeline against which all patient data will be aligned.
all_days = pd.DataFrame(
    {'DayRelativeToNearestHCT': range(min_day, max_day + 1)})

# Extract all unique 'PatientID' values from the 'merged_df' DataFrame.
# This list of unique patients will be iterated over to process data individually.
patients = merged_df['PatientID'].unique()

### Calculate Total Unique Patients

In [ ]:
# Calculate the total number of unique patients by getting the shape of the 'patients' array (which is a numpy array).
num_patients = patients.shape[0]  # Alternatively, len(patients) can be used.

# Print the total count of unique patients for informational purposes.
print(f"Total number of patients: {num_patients}")

### Initialize Storage for Aligned Data

In [ ]:
# Initialize an empty list named 'aligned_data'.
# This list will be used to temporarily store the processed and timeline-aligned DataFrame for each individual patient.
aligned_data = []

### Loop Through Patients to Align Data

In [ ]:
for patient in patients:
    # Filter the 'merged_df' to get all records pertaining to the current patient.
    patient_data = merged_df[merged_df['PatientID'] == patient]

    # Merge the 'all_days' timeline DataFrame with the current 'patient_data' using a 'left' join on 'DayRelativeToNearestHCT'.
    # This ensures that every day in the defined timeline is present for the patient, with original data filled in where available,
    # and NaN for days where the patient had no recorded data in the original merged_df.
    patient_timeline = all_days.merge(
        patient_data, on='DayRelativeToNearestHCT', how='left')

    # Explicitly assign the 'PatientID' to the newly created 'patient_timeline' DataFrame.
    # This is important because the 'merge' operation might not carry over the PatientID consistently across all rows if there were NaNs in the original data.
    patient_timeline['PatientID'] = patient

    # Append the fully processed and timeline-aligned DataFrame for the current patient to the 'aligned_data' list.
    aligned_data.append(patient_timeline)

### Concatenate Aligned Patient Data

In [ ]:
# Concatenate all individual patient timeline DataFrames stored in the 'aligned_data' list into a single, cohesive DataFrame.
# 'ignore_index=True' resets the index of the resulting DataFrame, creating a clean, continuous index.
final_df = pd.concat(aligned_data, ignore_index=True)

## 7. Handle Missing Values in Aligned Data

In [ ]:
# Handle missing values in 'MaxTemperature' using forward-fill (ffill).
# This method propagates the last valid observation forward to fill NaN values, assuming the temperature remains constant until a new measurement.
final_df['MaxTemperature'] = final_df['MaxTemperature'].ffill()

# Handle missing values in 'NeutrophilCount' using forward-fill (ffill).
# Similar to temperature, this assumes the neutrophil count remains constant until a new measurement is taken.
final_df['NeutrophilCount'] = final_df['NeutrophilCount'].ffill()

# Fill any remaining missing 'Consistency' values with the string 'Unknown'.
# This ensures that there are no NaN values in the categorical 'Consistency' column, making it ready for analysis.
final_df['Consistency'] = final_df['Consistency'].fillna('Unknown')

### Display Final Aligned DataFrame Shape

In [ ]:
# Display the dimensions (number of rows and columns) of the 'final_df' DataFrame.
# This helps in quickly verifying the size of the dataset after all preprocessing and alignment steps.
final_df.shape

### Save Processed Clinical Data

In [ ]:
# Save the 'final_df' DataFrame to a CSV file named 'processed_dataset.csv'.
# 'index=False' prevents pandas from writing the DataFrame's index as a column in the CSV file, keeping the output clean.
final_df.to_csv("processed_dataset.csv", index=False)

# Print a confirmation message to indicate that the processing is complete and the file has been saved.
print("Dataset processed and saved as 'processed_dataset.csv'")

## 8. Process Microbiome Data and Merge with Clinical Data

In [ ]:
import pandas as pd # Ensure pandas is imported, although it's likely already done at the top.

# Load the microbiome ASV (Amplicon Sequence Variant) counts data.
asv_counts = pd.read_csv("./data/raw/tblcounts_asv_melt.csv")

# Load the taxonomic classification for the ASVs.
taxonomy = pd.read_csv("./data/raw/tblASVtaxonomy_silva132_v4v5_filter.csv")

# Load the previously processed clinical data, which includes aligned patient information.
clinical_data = pd.read_csv("processed_dataset.csv") #Load previously generated & saved csv file.

# Load the ASV sample metadata, which contains PatientID, DayRelativeToNearestHCT, and Consistency.
asv_samples = pd.read_csv("./data/raw/tblASVsamples.csv")

# Step 1: Calculate Relative Abundance for each ASV within each SampleID.
# The 'transform' method is used to apply a function (x / x.sum()) to each group ('SampleID')
# and return a Series with the same index as the original DataFrame.
asv_counts['RelativeAbundance'] = asv_counts.groupby('SampleID')['Count'].transform(lambda x: x / x.sum())

# Step 2: Merge ASV counts with their taxonomic information.
# This step links each ASV to its Kingdom, Phylum, Class, Order, Family, and Genus.
asv_data = pd.merge(asv_counts, taxonomy, on='ASV', how='left')

# Step 3: Aggregate the relative abundances at the Genus level.
# This sums up the relative abundances of all ASVs belonging to the same Genus within each SampleID.
genus_data = asv_data.groupby(['SampleID', 'Genus'])['RelativeAbundance'].sum().reset_index()

# Step 4: Merge the Genus-level microbiome data with relevant sample metadata.
# This adds 'PatientID', 'DayRelativeToNearestHCT', and 'Consistency' to the genus data,
# which are crucial for linking to the clinical dataset.
genus_data = pd.merge(genus_data, asv_samples[['SampleID', 'PatientID', 'DayRelativeToNearestHCT', 'Consistency']], on='SampleID', how='left')

# Step 5: Merge the processed clinical data with the aggregated microbiome data.
# The merge is performed on 'PatientID' and 'DayRelativeToNearestHCT' to align the two datasets.
# A 'left' merge ensures all clinical data entries are retained, with microbiome data joined if available.
final_dataset = pd.merge(clinical_data, genus_data, on=['PatientID', 'DayRelativeToNearestHCT'], how='left')

# Step 6: Handle missing values in the 'final_dataset'.
# Missing values (NaN) in the microbiome features (after the left merge) are filled with 0.
# This implies that if a genus is not found for a particular patient on a specific day, its relative abundance is 0.
final_dataset.fillna(0, inplace=True)

# Step 7: Save the final integrated dataset to a CSV file.
# The 'HCT-Microbiome.csv' file now contains both clinical and microbiome data aligned by patient and day.
final_dataset.to_csv("HCT-Microbiome-Benchmark.csv", index=False)

# Print a confirmation message upon successful creation and saving of the final dataset.
print("Final dataset created: 'HCT-Microbiome-Benchmark.csv'")